# When to Pick an Overseas Player and When to Pick an Indian Player in the IPL Auction: A Data Analysis Project

**Problem Statement:**
In the Indian Premier League (IPL), every team can have a maximum of 8 overseas players in the squad and only 4 overseas players in the playing 11. Because of this rule, teams must carefully decide whether to spend money on an overseas player or an Indian player during the auction.
This project aims to study past IPL auction data, player prices, performance statistics, and team results to find clear patterns. The goal is to understand in which situations it is better to pick an overseas player and in which situations an Indian player is the smarter choice.

---

### Step 1: Load Year Constants & Historical Dataset (DBMS + Saaty AHP + TOPSIS + ML Regression)
When a `year` (`2018` to `2026`) is selected, the system automatically loads:
1. `players_list` and `players_with_prices` from `ipl_auctions_2018_2026.csv`
2. `total_purse` for that season (`Rs. 80 Cr` in 2018 up to `Rs. 125 Cr` in 2026)
3. Pre-auction **TOPSIS** performance scores (using Saaty's AHP weights on strictly prior IPL deliveries) and **Multiple Linear Regression** fair value estimates.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from interactive_auction_model import (
    PURSE_BY_YEAR,
    get_year_constants,
    evaluate_auction_pick,
    interactive_auction_model,
    year_slider,
    sold_to_same_widget,
    sold_to_other_widget,
    player_on_hammer_dropdown,
    interactive_output,
    controls
)

data_2024 = get_year_constants(2024)
print(f"Loaded IPL {data_2024['year']}: {len(data_2024['players_list'])} players | Purse = Rs.{data_2024['total_purse']/100:.0f} Crore")
display(data_2024['pool'][['Player Name', 'Player Nationality', 'Player Role', 'Base_Clean', 'Eval_Price_Lakhs', 'TOPSIS', 'ML_Fair_Price_Lakhs']].head(8))

**Analysis:** Notice that all player scores (`TOPSIS`) and fair value predictions (`ML_Fair_Price_Lakhs`) for `2024` are calculated strictly from pre-2024 data (`2019-2023`), preventing any data leakage. The season purse (`Rs. 100 Crore` = `10,000 Lakhs`) and complete player list with prices are loaded automatically.

### Step 2: Interactive Auction Decision & Break-Even Max Bid Slider (`PuLP` Linear Programming)
Use the controls below to pick:
- **Auction Year** (`2018` to `2026`)
- **Sold to Same Team** (players already on your roster)
- **Sold to Other Team** (players already bought/retained by rival teams)
- **Player on Hammer** (the player currently being bid on)

The **PuLP** 0-1 Integer Linear Program enforces all IPL squad rules (`18-25` players, `10-25` Indians, `0-8` Overseas, `75%-100%` Purse Spend) and classifies the pick into `"always overseas"`, `"always Indian"`, or `"overseas due to constraints"`, along with the **exact break-even maximum bid price**.

In [ ]:
display(controls, interactive_output)

**Analysis of Interactive Output:**
- If an overseas player is selected in both the **Unconstrained** core LP and the **Constrained** full-squad LP, the model recommends **`"always overseas"`**.
- If an overseas player is only picked because rival teams locked up top Indian alternatives or because of the `18-player` / `75% purse` constraints, the model outputs **`"overseas due to constraints"`**.
- If the `8-overseas` quota is full or Indian alternatives yield higher total TOPSIS score per rupee, the model outputs **`"always Indian"`** (and sets the overseas player's max bid to `Rs. 0 Lakhs`).
- Finally, binary search with PuLP finds the exact **break-even price** (`upto what price can pick?`) above which bidding hurts the team's overall squad score.

### Step 3: Batch Verification on Sample IPL 2024 Auction Cases
Below we run a reproducible test across representative Indian and Overseas players ahead of the IPL 2024 auction.

In [ ]:
import pandas as pd

pool24 = data_2024['pool']
same_team_demo = [
    'MS Dhoni', 'Ravindra Jadeja', 'Ruturaj Gaikwad', 'Shivam Dube',
    'Deepak Chahar', 'Matheesha Pathirana', 'Devon Conway', 'Moeen Ali'
]
other_team_demo = list(set(pool24[pool24['Is_Retained'] == 1]['Player Name']) - set(same_team_demo))

rows = []
for p in ['Shardul Thakur', 'Lockie Ferguson', 'Pat Cummins', 'Nandre Burger', 'Travis Head', 'Shashank Singh']:
    r = evaluate_auction_pick(2024, same_team_demo, other_team_demo, p)
    rows.append({
        'Player': r['Player'],
        'Origin': r['Origin'],
        'Role': r['Role'],
        'TOPSIS': r['TOPSIS_Score'],
        'Base (L)': r['Base_Price_Lakhs'],
        'ML Fair (L)': r['ML_Fair_Price_Lakhs'],
        'Decision': r['Decision'],
        'Max Bid (L)': r['Max_Bid_Lakhs']
    })

display(pd.DataFrame(rows))

### Conclusion
1. **Pick Indian Players** for top/middle-order batting (`203 Runs/Cr` vs `99 Runs/Cr` Overseas) and squad depth (`3x-10x` cheaper median auction prices, lower unsold/quota risk). Winning teams (`>=50%` win rate) spend **less overall** on overseas players (`38.6%` vs `42.7%` of purse).
2. **Pick Overseas Players** primarily for **strike/death bowling** (winning teams spend `13.9%` vs `10.1%` on overseas bowlers, `r = +0.182`), **death-over hitting** (`157.1 SR` vs `147.6 SR`), and **open-auction WK-Batters** (`Rs. 3.88 Cr` avg vs `Rs. 4.68 Cr` for Indian WK-Batters), while capping every bid at the **PuLP break-even price**.